## Source necessary dependencies

In [41]:
import os
#print current directory for sanity check
print("Present working directory:", os.getcwd())
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim

Present working directory: /home/nd329/causal_forecast


## Default configurations for the VAE + LSTM forecast in causal panel model

In [42]:
DEFAULT_N = 64
DEFAULT_H = 3
DEFAULT_R = 3
DEFAULT_MISSING_RATE = 0.30
DEFAULT_REPS = 50
DEFAULT_N_JOBS = 50   # you asked for 50 parallel processes

## Data generating processes

- **DGP 1**: Non-linear latent structure-to-observation map and linear latent dynamics.
\begin{align*}
Y_{i,t} = & \Lambda_i^\top F_t + 0.5 \sin(\Lambda_i^\top F_t) + \varepsilon_{i,t},\\
F_t = & A F_{t-1} + \eta_t.
\end{align*}

- **DGP 2**: Linear latent structure-to-observation map ($r = 2$), non-linear latent dynamics.
\begin{align*}
Y_t = & \Lambda F_t + \varepsilon_t,\\
F_{t,1} = & 0.65 F_{t-1,1} + 0.15 \sin(F_{t-1,2}) + \eta_{t, 1},\\
F_{t,2} = & 0.55 F_{t-1,1} + 0.10 F_{t-1,2}^2 + \eta_{t, 2}.
\end{align*}

- **DGP 3**: Non-linear latent structure-to-observation map ($r = 2$), non-linear latent dynamics.
\begin{align*}
Y_{i,t} = & \Lambda_i^\top F_t + 0.4 \sin(\Lambda_i^\top F_t) + \varepsilon_{i,t},\\
F_{t,1} = & 0.65 F_{t-1,1} + 0.15 \sin(F_{t-1,2}) + \eta_{t, 1},\\
F_{t,2} = & 0.55 F_{t-1,1} + 0.10 F_{t-1,2}^2 + \eta_{t, 2}.
\end{align*}

In [44]:
# ============================================================
# DGP-1
# ============================================================

def generate_dgp1(T_total=515, N=64, r=3, noise_std=0.1, seed=123):
    rng = np.random.default_rng(seed)

    A = np.array([
        [0.7, 0.1, 0.0],
        [0.0, 0.6, 0.1],
        [0.0, 0.0, 0.5]
    ])[:r, :r]

    z = np.zeros((T_total, r))
    for t in range(1, T_total):
        z[t] = A @ z[t-1] + rng.normal(scale=0.2, size=r)

    Wlin = rng.normal(size=(N, r))
    Vnon = rng.normal(size=(N, r))

    Y = np.zeros((N, T_total))
    for t in range(T_total):
        Y[:, t] = (
            Wlin @ z[t]
            + 0.5 * np.sin(Vnon @ z[t])
            + rng.normal(scale=noise_std, size=N)
        )

    params = {
        "type": 1,
        "A": A,
        "Wlin": Wlin,
        "Vnon": Vnon,
        "noise_std": noise_std,
        "r": r,
        "N": N
    }
    return Y, z, params

# ============================================================
# DGP-2
# ============================================================
def generate_dgp2(T_total=515, N=64, r=2, noise_std=0.1, seed=123):
    rng = np.random.default_rng(seed)
    z = np.zeros((T_total, r))

    for t in range(1, T_total):
        z_prev = z[t-1].copy()
        z[t, 0] = 0.6 * z_prev[0] + 0.2 * np.sin(z_prev[1]) + rng.normal(scale=0.15)
        z[t, 1] = 0.5 * z_prev[1] + 0.15 * (z_prev[0] ** 2) + rng.normal(scale=0.15)

    Lambda = rng.normal(size=(N, r))
    Y = Lambda @ z.T + rng.normal(scale=noise_std, size=(N, T_total))

    params = {
        "type": 2,
        "Lambda": Lambda,
        "noise_std": noise_std,
        "r": r,
        "N": N
    }
    return Y, z, params


# ============================================================
# DGP-3
# ============================================================
def generate_dgp3(T_total=515, N=64, r=2, noise_std=0.1, seed=123):
    rng = np.random.default_rng(seed)
    z = np.zeros((T_total, r))

    for t in range(1, T_total):
        z_prev = z[t-1].copy()
        z[t, 0] = 0.65 * z_prev[0] + 0.15 * np.sin(z_prev[1]) + rng.normal(scale=0.15)
        z[t, 1] = 0.55 * z_prev[1] + 0.10 * (z_prev[0] ** 2) + rng.normal(scale=0.15)

    Wlin = rng.normal(size=(N, r))
    Vnon = rng.normal(size=(N, r))

    Y = np.zeros((N, T_total))
    for t in range(T_total):
        Y[:, t] = (
            Wlin @ z[t]
            + 0.4 * np.sin(Vnon @ z[t])
            + rng.normal(scale=noise_std, size=N)
        )

    params = {
        "type": 3,
        "Wlin": Wlin,
        "Vnon": Vnon,
        "noise_std": noise_std,
        "r": r,
        "N": N
    }
    return Y, z, params

# ============================================================
# Generating for all DGPs
# ============================================================
def generate_panel(dgp_id, T_total, N, seed):
    if dgp_id == 1:
        return generate_dgp1(T_total=T_total, N=N, seed=seed)
    elif dgp_id == 2:
        return generate_dgp2(T_total=T_total, N=N, seed=seed)
    elif dgp_id == 3:
        return generate_dgp3(T_total=T_total, N=N, seed=seed)
    else:
        raise ValueError("dgp_id must be 1, 2, or 3.")

## Missingness pattern generation
The missigness patters in generated as MCAR with observation probability $p = 0.70$.

In [45]:
# ============================================================
# 2. MISSINGNESS
# ============================================================

def generate_W_mcar(Y_train, missing_rate=0.30, seed=123):
    """
    W[i,t] = 1 if observed, 0 if missing
    MCAR missingness
    """
    rng = np.random.default_rng(seed)
    N, T = Y_train.shape
    W = (rng.uniform(size=(N, T)) > missing_rate).astype(np.float32)
    return W


def apply_W(Y_train, W):
    """
    observed training matrix with missing entries zero-filled
    """
    return Y_train * W

## Function for the non-linear factor estimator: Masked autoencoder

In [46]:
# ============================================================
# 3. NONLINEAR FACTOR ESTIMATOR: MASKED AUTOENCODER
# ============================================================

class MaskedAutoencoder(nn.Module):
    def __init__(self, N, r, hidden=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(2 * N, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, r)
        )
        self.decoder = nn.Sequential(
            nn.Linear(r, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, N)
        )

    def encode(self, y, W):
        x = torch.cat([y * W, W], dim=-1)
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, y, W):
        z = self.encode(y, W)
        yhat = self.decode(z)
        return yhat, z


def train_masked_autoencoder(Y_obs, W, r=2, hidden=128, epochs=300, lr=1e-3, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    N, T = Y_obs.shape
    model = MaskedAutoencoder(N=N, r=r, hidden=hidden).to(device)

    y_data = torch.tensor(Y_obs.T, dtype=torch.float32).to(device)   # T x N
    w_data = torch.tensor(W.T, dtype=torch.float32).to(device)       # T x N

    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        yhat, z = model(y_data, w_data)
        loss = ((w_data * (y_data - yhat) ** 2).sum() / w_data.sum())
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        zhat = model.encode(y_data, w_data).cpu().numpy()   # T x r

    return model, zhat

## Non-linear latent factor forecaster: MLP and LSTM
Below are the functions for latent factor dynamics learner and forecaster with MLP and LSTM with PyTorch functions.

In [47]:
# ============================================================
# 4. NONLINEAR LATENT FORECASTERS: MLP and LSTM
# ============================================================

def build_supervised_latent_data(z, L=5):
    T, r = z.shape
    X, Y = [], []
    for t in range(L, T):
        X.append(z[t-L:t])
        Y.append(z[t])
    return np.array(X), np.array(Y)


class LatentMLP(nn.Module):
    def __init__(self, r, L=5, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(L * r, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, r)
        )

    def forward(self, x):
        return self.net(x)


def train_latent_mlp(zhat, L=5, hidden=64, epochs=200, lr=1e-3, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X, Y = build_supervised_latent_data(zhat, L=L)
    n, L_, r = X.shape

    X = X.reshape(n, L_ * r)
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    Y_t = torch.tensor(Y, dtype=torch.float32).to(device)

    model = LatentMLP(r=r, L=L, hidden=hidden).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_t)
        loss = loss_fn(pred, Y_t)
        loss.backward()
        optimizer.step()

    return model


def forecast_latent_mlp(model, zhat, H=3, L=5, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    history = zhat.copy().tolist()
    preds = []
    model.eval()
    with torch.no_grad():
        for _ in range(H):
            x = np.array(history[-L:]).reshape(1, -1)
            x_t = torch.tensor(x, dtype=torch.float32).to(device)
            z_next = model(x_t).cpu().numpy()[0]
            preds.append(z_next)
            history.append(z_next)
    return np.array(preds)


class LatentLSTM(nn.Module):
    def __init__(self, r, hidden=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=r, hidden_size=hidden, batch_first=True)
        self.fc = nn.Linear(hidden, r)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


def train_latent_lstm(zhat, L=5, hidden=64, epochs=200, lr=1e-3, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    X, Y = build_supervised_latent_data(zhat, L=L)
    X_t = torch.tensor(X, dtype=torch.float32).to(device)
    Y_t = torch.tensor(Y, dtype=torch.float32).to(device)

    r = zhat.shape[1]
    model = LatentLSTM(r=r, hidden=hidden).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        pred = model(X_t)
        loss = loss_fn(pred, Y_t)
        loss.backward()
        optimizer.step()

    return model


def forecast_latent_lstm(model, zhat, H=3, L=5, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    history = zhat.copy().tolist()
    preds = []

    model.eval()
    with torch.no_grad():
        for _ in range(H):
            x = np.array(history[-L:])[None, :, :]
            x_t = torch.tensor(x, dtype=torch.float32).to(device)
            z_next = model(x_t).cpu().numpy()[0]
            preds.append(z_next)
            history.append(z_next)
    return np.array(preds)


## Deocder of the forecasted latent factors

In [48]:
# ============================================================
# 5. DECODE LATENT FORECASTS TO PANEL FORECASTS
# ============================================================

def decode_forecasts(ae_model, z_forecast, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    z_t = torch.tensor(z_forecast, dtype=torch.float32).to(device)
    ae_model.eval()
    with torch.no_grad():
        yhat = ae_model.decode(z_t).cpu().numpy()   # H x N
    return yhat.T   # N x H


## Calculator for the oracle conditional means

In [ ]:
# ============================================================
# 6. ORACLE CONDITIONAL MEAN C0 = E[Y_{T+1:T+H} | present]
#    For DGP 2 and 3, computed by Monte Carlo from true z_T
# ============================================================

def latent_step_dgp1(z_prev, params, rng):
    A = params["A"]
    r = params["r"]
    z_next = A @ z_prev + rng.normal(scale=0.2, size=r)
    return z_next

def latent_step_dgp2(z_prev, rng):
    z_next = np.zeros_like(z_prev)
    z_next[0] = 0.6 * z_prev[0] + 0.2 * np.sin(z_prev[1]) + rng.normal(scale=0.15)
    z_next[1] = 0.5 * z_prev[1] + 0.15 * (z_prev[0] ** 2) + rng.normal(scale=0.15)
    return z_next

def latent_step_dgp3(z_prev, rng):
    z_next = np.zeros_like(z_prev)
    z_next[0] = 0.65 * z_prev[0] + 0.15 * np.sin(z_prev[1]) + rng.normal(scale=0.15)
    z_next[1] = 0.55 * z_prev[1] + 0.10 * (z_prev[0] ** 2) + rng.normal(scale=0.15)
    return z_next

def obs_mean_from_z_dgp1(z, params):
    return params["Wlin"] @ z + 0.5 * np.sin(params["Vnon"] @ z)

def obs_mean_from_z_dgp2(z, params):
    return params["Lambda"] @ z

def obs_mean_from_z_dgp3(z, params):
    return params["Wlin"] @ z + 0.4 * np.sin(params["Vnon"] @ z)

def mc_conditional_mean_C0(dgp_id, z_T, params, H=3, n_mc=2000, seed=999):
    """
    Returns N x H matrix:
    C0[:, h-1] approx E[Y_{T+h} | z_T]
    """
    rng = np.random.default_rng(seed)
    N = params["N"]
    C0 = np.zeros((N, H))

    for _ in range(n_mc):
        z_curr = z_T.copy()
        for h in range(H):
            if dgp_id == 1:
                z_curr = latent_step_dgp1(z_curr, params, rng)
                y_mean = obs_mean_from_z_dgp1(z_curr, params)
            elif dgp_id == 2:
                z_curr = latent_step_dgp2(z_curr, rng)
                y_mean = obs_mean_from_z_dgp2(z_curr, params)
            elif dgp_id == 3:
                z_curr = latent_step_dgp3(z_curr, rng)
                y_mean = obs_mean_from_z_dgp3(z_curr, params)
            else:
                raise ValueError("dgp_id must be 1, 2, or 3.")
            C0[:, h] += y_mean

    C0 /= n_mc
    return C0

## Single repplicate of the forecast experiment

In [50]:
# ============================================================
# 7. SINGLE-REPLICATE PIPELINE
# ============================================================

def ensure_dirs(base_dir, dgp_id):
    y_dir = Path(base_dir) / f"NL_DGP{dgp_id}" / "Y_files"
    c0_dir = Path(base_dir) / f"NL_DGP{dgp_id}" / "C0_files"
    w_dir = Path(base_dir) / f"NL_DGP{dgp_id}" / "W_files"

    y_dir.mkdir(parents=True, exist_ok=True)
    c0_dir.mkdir(parents=True, exist_ok=True)
    w_dir.mkdir(parents=True, exist_ok=True)

    return y_dir, c0_dir, w_dir


def save_csvs(Y_train, C0, W, dgp_id, N, T, iter_id, base_dir="data_files"):
    y_dir, c0_dir, w_dir = ensure_dirs(base_dir, dgp_id)

    y_file = y_dir / f"NL_DGP{dgp_id}_Y_N{N}_T{T}_iter{iter_id}.csv"
    c0_file = c0_dir / f"NL_DGP{dgp_id}_C0_N{N}_T{T}_iter{iter_id}.csv"
    w_file = w_dir / f"NL_DGP{dgp_id}_W_N{N}_T{T}_iter{iter_id}.csv"

    pd.DataFrame(Y_train).to_csv(y_file, header=False, index=False)
    pd.DataFrame(C0).to_csv(c0_file, header=False, index=False)
    pd.DataFrame(W).to_csv(w_file, header=False, index=False)


def run_one_replicate(
    iter_id,
    dgp_id=2,
    N=64,
    T=128,
    H=3,
    r=None,
    missing_rate=0.30,
    dyn_model="lstm",   # "lstm" or "mlp"
    lookback_L=5,
    ae_hidden=128,
    dyn_hidden=64,
    ae_epochs=300,
    dyn_epochs=200,
    lr=1e-3,
    n_mc_c0=2000,
    save_files=True,
    base_dir="data_files"
):
    """
    One replicate:
    - generate full panel of length T+H
    - keep first T columns as training Y
    - generate W on training set
    - fit masked AE + nonlinear dynamics model
    - forecast H=3 future steps
    - compute C0 = E[Y_future | present] via MC
    - save Y_train, C0, W
    - return MSE averaged only over first 32 rows
    """
    if r is None:
        r = 2 if dgp_id in [2, 3] else 3

    seed = 10000 + iter_id
    T_total = T + H

    # generate full data
    Y_full, z_full, params = generate_panel(dgp_id=dgp_id, T_total=T_total, N=N, seed=seed)

    Y_train = Y_full[:, :T]         # N x T
    z_train = z_full[:T, :]         # T x r
    z_T = z_full[T - 1, :]          # true latent state at time T
    Y_future_true = Y_full[:, T:T+H]

    # missingness on training data only
    W = generate_W_mcar(Y_train, missing_rate=missing_rate, seed=seed + 1)
    Y_obs = apply_W(Y_train, W)

    # fit nonlinear factor estimator
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ae_model, zhat = train_masked_autoencoder(
        Y_obs, W, r=r, hidden=ae_hidden, epochs=ae_epochs, lr=lr, device=device
    )

    # fit nonlinear dynamics
    if dyn_model.lower() == "lstm":
        dyn = train_latent_lstm(
            zhat, L=lookback_L, hidden=dyn_hidden, epochs=dyn_epochs, lr=lr, device=device
        )
        z_fore = forecast_latent_lstm(dyn, zhat, H=H, L=lookback_L, device=device)
    elif dyn_model.lower() == "mlp":
        dyn = train_latent_mlp(
            zhat, L=lookback_L, hidden=dyn_hidden, epochs=dyn_epochs, lr=lr, device=device
        )
        z_fore = forecast_latent_mlp(dyn, zhat, H=H, L=lookback_L, device=device)
    else:
        raise ValueError("dyn_model must be 'lstm' or 'mlp'.")

    Y_fore = decode_forecasts(ae_model, z_fore, device=device)   # N x H

    # oracle conditional mean
    C0 = mc_conditional_mean_C0(dgp_id, z_T, params, H=H, n_mc=n_mc_c0, seed=seed + 2)

    # average only over first 32 rows
    row_end = min(32, N)
    mse = np.mean((Y_fore[:row_end, :] - C0[:row_end, :]) ** 2)

    # optionally save files
    if save_files:
        save_csvs(Y_train, C0, W, dgp_id=dgp_id, N=N, T=T, iter_id=iter_id, base_dir=base_dir)

    return {
        "iter": iter_id,
        "dgp_id": dgp_id,
        "N": N,
        "T": T,
        "H": H,
        "dyn_model": dyn_model,
        "mse_first32": mse
    }

## Many parallelly replicated experiment and calculating average forecast error

In [51]:
# ============================================================
# 8. MANY REPLICATES, PARALLEL
# ============================================================

def run_many_replicates(
    dgp_id=2,
    N=64,
    T=128,
    H=3,
    reps=50,
    n_jobs=50,
    dyn_model="lstm",
    base_dir="data_files"
):
    results = []

    with ProcessPoolExecutor(max_workers=n_jobs) as ex:
        futures = []
        for iter_id in range(1, reps + 1):
            futures.append(
                ex.submit(
                    run_one_replicate,
                    iter_id=iter_id,
                    dgp_id=dgp_id,
                    N=N,
                    T=T,
                    H=H,
                    dyn_model=dyn_model,
                    base_dir=base_dir
                )
            )

        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"DGP {dgp_id}, N={N}, T={T}"):
            results.append(fut.result())

    results = sorted(results, key=lambda x: x["iter"])
    df = pd.DataFrame(results)
    avg_mse = df["mse_first32"].mean()

    return df, avg_mse


## Function for executing the parallel runs across multiple grids

In [52]:
# ============================================================
# 9. RUN GRID OVER T = 32, 64, 128, 256, 512
# ============================================================

def run_grid_for_dgp(
    dgp_id=2,
    N=64,
    T_list=(32, 64, 128, 256, 512),
    H=3,
    reps=50,
    n_jobs=50,
    dyn_model="lstm",
    base_dir="data_files",
    summary_csv_name=None
):
    all_tables = []

    for T in tqdm(T_list, desc=f"Grid for DGP {dgp_id}, model={dyn_model}"):
        df, avg_mse = run_many_replicates(
            dgp_id=dgp_id,
            N=N,
            T=T,
            H=H,
            reps=reps,
            n_jobs=n_jobs,
            dyn_model=dyn_model,
            base_dir=base_dir
        )
        df["avg_mse_over_50_reps"] = avg_mse
        all_tables.append(df)

        print(f"DGP {dgp_id}, N={N}, T={T}, avg MSE over 50 reps = {avg_mse:.6f}")

    out = pd.concat(all_tables, ignore_index=True)

    if summary_csv_name is None:
        summary_csv_name = f"summary_NL_DGP{dgp_id}_N{N}_{dyn_model}.csv"

    out.to_csv(summary_csv_name, index=False)
    return out

## Example usage

In [55]:
# ============================================================
# 10. EXAMPLE USAGE
# ============================================================

if __name__ == "__main__":
    # ----------------------
    # custom settings
    # ----------------------
    dgp_id = 3         # 1, 2 or 3
    N = 64
    T_list = [32, 64, 128, 256, 512]
    dyn_model = "lstm"  # or "mlp"

    summary = run_grid_for_dgp(
        dgp_id=dgp_id,
        N=N,
        T_list=T_list,
        H=3,
        reps=50,
        n_jobs=50,
        dyn_model=dyn_model,
        base_dir="data_files"
    )

    print(summary.groupby(["dgp_id", "N", "T"])["mse_first32"].mean())

Grid for DGP 3, model=lstm:  20%|██        | 1/5 [00:04<00:16,  4.11s/it]

DGP 3, N=64, T=32, avg MSE over 50 reps = 0.073038


Grid for DGP 3, model=lstm:  40%|████      | 2/5 [00:08<00:13,  4.56s/it]

DGP 3, N=64, T=64, avg MSE over 50 reps = 0.049080


Grid for DGP 3, model=lstm:  60%|██████    | 3/5 [00:14<00:09,  4.92s/it]

DGP 3, N=64, T=128, avg MSE over 50 reps = 0.028185


Grid for DGP 3, model=lstm:  80%|████████  | 4/5 [00:20<00:05,  5.47s/it]

DGP 3, N=64, T=256, avg MSE over 50 reps = 0.008768


Grid for DGP 3, model=lstm: 100%|██████████| 5/5 [00:29<00:00,  5.92s/it]

DGP 3, N=64, T=512, avg MSE over 50 reps = 0.003700
dgp_id  N   T  
3       64  32     0.073038
            64     0.049080
            128    0.028185
            256    0.008768
            512    0.003700
Name: mse_first32, dtype: float64
